In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch00-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "00-04-reading-guide.py"

# 這本書怎麼用

先從上到下執行。下格將分析程式寫入檔案，最後一格才使用安裝好的環境執行它。

資料為 DeepChem 載入的 ESOL，保留全部紀錄，不訓練模型、不生成假量測值。原始研究：[Delaney 2004](https://doi.org/10.1021/ci034243x)；資料彙整：[MoleculeNet](https://doi.org/10.1039/C7SC02664A)。

Colab 使用 CPU 即可。若看不到 2026.07，請先核對書籍環境說明，勿忽略版本檢查。

<!-- concept-ch00-05:start -->
讀到一張預測結果圖時，可以逐項把圖上的數字連回資料。

| 圖上看到什麼 | 回正文找什麼 | 這一步釐清什麼 |
|---|---|---|
| 橫軸寫 logS | 定義為溶解度 mol/L 的常用對數 | 負值不是負濃度 |
| 每個點是一個分子 | 對應的 SMILES 與資料列 | 哪些點屬於相近結構 |
| 圖說寫測試集誤差 | 切分方式與測試資料 | 誤差適用於哪批分子 |

這張表示範實際的讀圖動作。以第一列為例，先知道 logS 的定義，再判斷圖上相差一單位代表什麼。

<!-- concept-ch00-05:end -->

## 怎麼讀篩選程式

`table` 是存放資料的表格，`table["MW"]` 表示取出名為 `MW` 的整個欄位；方括號裡的引號表示這是一個欄位名稱。將這欄與 200 比較後，每一列都會得到 `True`（符合）或 `False`（不符合）。例如 MW 為 180 的分子，`small_enough` 就是 `True`。

`parse_ok` 是前面讀取結構時建立的檢查欄：成功讀入 SMILES 就記為 `True`，失敗則記為 `False`。`weight` 則是資料來源提供的標籤權重，在這份資料中，大於零表示該列有可用的實測值；它不是分子的重量。

最後用 `&` 把四個條件連起來，得到每列是否入選的 `mask`。`table.loc[mask]` 依這份檢查結果挑出標成 `True` 的資料列，`.copy()` 再把結果複製成另一張表 `selected`，方便後續使用。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import json
import numpy as np
import pandas as pd
import os
# 使用適合輸出圖檔的繪圖後端，避免沿用 Colab 的顯示設定。
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
import deepchem as dc

# ESOL 真實水溶解度；原始研究 DOI:10.1021/ci034243x。
# 本章只讀取、核對與篩選，不訓練模型，因此不切分資料；不轉換標籤。
tasks, parts, transformers = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False)
data = parts[0]
assert len(tasks) == 1 and not transformers
mols = [Chem.MolFromSmiles(str(s)) for s in data.ids]
table = pd.DataFrame({"row_id": np.arange(len(data)), "smiles": data.ids,
    "logS": data.y[:, 0], "weight": data.w[:, 0],
    "parse_ok": [m is not None for m in mols]})
table["canonical_smiles"] = [Chem.MolToSmiles(m) if m is not None else None for m in mols]
table["MW"] = [Descriptors.MolWt(m) if m is not None else np.nan for m in mols]
book = Path("../..").resolve() if Path("../../_quarto.yml").is_file() else Path.cwd()
figures = book / "figures"
figures.mkdir(exist_ok=True)
outputs = Path("outputs")
outputs.mkdir(exist_ok=True)
print("Python / RDKit:", __import__("sys").version.split()[0], __import__("rdkit").__version__)
print("Task:", tasks[0], "rows:", len(table))
# 這是閱讀練習的任意門檻，不是藥物篩選規範；logS 為已量測值。
# 先分別檢查四個條件，每個條件都會得到一欄 True 或 False。
small_enough = table["MW"] <= 200       # 莫耳質量不超過 200 g/mol
soluble_enough = table["logS"] >= -2   # 實測 logS 不低於 −2
has_structure = table["parse_ok"]     # 結構是否成功讀入
has_measurement = table["weight"] > 0 # 這筆紀錄是否有可用的實測值

# 只有四個條件都成立的資料列，才會被選中。
mask = small_enough & soluble_enough & has_structure & has_measurement
selected = table.loc[mask].copy()
print("Selected records:", len(selected))
print(selected[["row_id","smiles","MW","logS"]].head(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(7,4.5))
ax.scatter(table["MW"],table["logS"],s=12,c="#bbb",label="All records")
ax.scatter(selected["MW"],selected["logS"],s=15,c="#287d8e",label="Selected by measured values")
ax.axvline(200,ls="--",c="#555");ax.axhline(-2,ls="--",c="#555")
ax.set(xlabel="Calculated molecular weight (g/mol)",ylabel="Measured logS [log10(mol/L)]")
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(figures / "00-04-reading.png",dpi=170);plt.close(fig)


In [ ]:
from IPython.display import Image, display
run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "00-04-reading.png")))